In [11]:
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

client = OpenAI(
    base_url="https://ollama.com/v1",
    api_key=os.getenv("OLLAMA_API_KEY")
)

MODEL = "gpt-oss:120b"

In [2]:
system_message = """
You are a helpful assistant for a bookstore called BookStoreAI.
Give short, polite answers (keep it short).
Be accurate.
If you don’t know something, say so.
"""

In [3]:
DB = "books.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS books (
            title TEXT PRIMARY KEY,
            price REAL
        )
    """)
    conn.commit()

In [4]:
def set_book_price(title, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO books (title, price)
            VALUES (?, ?)
            ON CONFLICT(title)
            DO UPDATE SET price = ?
        """, (title.lower(), price, price))
        conn.commit()

# Add sample books
books = {
    "harry potter": 499,
    "atomic habits": 399,
    "the alchemist": 299,
    "deep work": 450
}

for title, price in books.items():
    set_book_price(title, price)

In [5]:
def get_book_price(title):
    print(f"DATABASE TOOL CALLED: Getting price for {title}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM books WHERE title = ?", (title.lower(),))
        result = cursor.fetchone()
        
        if result:
            return f"The price of '{title}' is ₹{result[0]}"
        else:
            return "No price data available for this book"

In [6]:
book_price_function = {
    "name": "get_book_price",
    "description": "Get the price of a book from the bookstore database.",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "The title of the book the customer wants"
            }
        },
        "required": ["title"],
        "additionalProperties": False
    }
}

In [7]:
tools = [
    {
        "type": "function",
        "function": book_price_function
    }
]

In [8]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_book_price":
            args = json.loads(tool_call.function.arguments)

            title = args.get("title")
            price_details = get_book_price(title)

            responses.append({
                "role" : "tool",
                "content" : price_details,
                "tool_call_id" : tool_call.id
            })
    
    return responses

In [9]:
def chat(message, history):

    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":

        message = response.choices[0].message

        responses = handle_tool_calls(message)

        messages.append(message)
        messages.extend(responses)

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="📚 BookStoreAI Assistant"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for The Alchemist
DATABASE TOOL CALLED: Getting price for Atomic habits
DATABASE TOOL CALLED: Getting price for Deep work
DATABASE TOOL CALLED: Getting price for Where the Crawdads Sing
